In [1]:
import sys, os

In [22]:
# notebook parameters, will be replaced by script arguments
application_name = "My Application"
s3a_access_key = os.environ.get("s3a_access_key")
s3a_secret_key = os.environ.get("s3a_secret_key")
input_file_1 = "s3a://uga-data-lake/sensor-streamer-test/android-sensor-*/*/*/*"
output_file_1 = "s3a://uga-data-lake/sensor-streamer-test/analytics"

# The demo will use the app [Android Sensor Cloud Streamer](https://androidsensor.ai.studio/)

In [3]:
import pyspark
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
import pyspark.sql.types as T
import pyspark.sql.functions as F
pyspark.__version__

'4.1.2'

In [4]:
# test if the script is running as a standalone program or a notebook
if __name__ == '__main__' and '__file__' in globals():
    print("Running as a script")
    application_name = sys.argv[1]
    s3a_access_key = sys.argv[2]
    s3a_secret_key = sys.argv[3]
    input_file_1 = sys.argv[4]
    output_file_1 = sys.argv[5]
    interactive = False
else:
    interactive = True
    print("Running as a notebook")
    pv = !{sys.executable} --version
    print("Python version ", pv)
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    print("PySpark version ", pyspark.__version__)
    jv = !java --version
    print("Java version ", jv)

Running as a notebook
Python version  ['Python 3.12.13']
PySpark version  4.1.2
Java version  ['openjdk 17.0.19 2026-04-21', 'OpenJDK Runtime Environment (build 17.0.19+10-1-deb12u2-Debian)', 'OpenJDK 64-Bit Server VM (build 17.0.19+10-1-deb12u2-Debian, mixed mode, sharing)']


In [5]:
def show(df):
    if interactive:
        df.show()

In [6]:
spark = (SparkSession.builder.appName(application_name)
         .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
         .config('spark.hadoop.fs.s3a.access.key', s3a_access_key)
         .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
         .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-2.amazonaws.com")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a21632b0-227a-4247-a5ec-5516007f3b34;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 176ms :: artifacts dl 6ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 from central in [default]
	software.amazon.s3.analyt

In [23]:
input_data = spark.read.option("multiLine", True).json(input_file_1)
input_data.show(n=3, vertical=True, truncate=False)

-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 activeSensor

In [24]:
input_data.select("deviceId").distinct().show(truncate=False)

+------------------------+
|deviceId                |
+------------------------+
|android-sensor-de3d-43ab|
|android-sensor-hanisaf  |
+------------------------+



In [26]:
input_data.isStreaming

False

In [28]:
accel_stats_schema = T.StructType([
    T.StructField("sampleCount", T.IntegerType(), True),
    T.StructField("meanX", T.DoubleType(), True),
    T.StructField("meanY", T.DoubleType(), True),
    T.StructField("meanZ", T.DoubleType(), True),
    T.StructField("magnitudeMax", T.DoubleType(), True),
    T.StructField("magnitudeMin", T.DoubleType(), True),
    T.StructField("magnitudeVariance", T.DoubleType(), True),
    T.StructField("potentialAnomaly", T.BooleanType(), True)
])

accel_current_schema = T.StructType([
    T.StructField("x", T.DoubleType(), True),
    T.StructField("y", T.DoubleType(), True),
    T.StructField("z", T.DoubleType(), True),
    T.StructField("magnitude", T.DoubleType(), True)
])

gyro_stats_schema = T.StructType([
    T.StructField("sampleCount", T.IntegerType(), True),
    T.StructField("meanAlpha", T.DoubleType(), True),
    T.StructField("meanBeta", T.DoubleType(), True),
    T.StructField("meanGamma", T.DoubleType(), True),
    T.StructField("peakRotationRate", T.DoubleType(), True)
])

gyro_current_schema = T.StructType([
    T.StructField("alpha", T.DoubleType(), True),
    T.StructField("beta", T.DoubleType(), True),
    T.StructField("gamma", T.DoubleType(), True),
    T.StructField("rotationMagnitude", T.DoubleType(), True)
])

sensor_payload_schema = T.StructType([
    T.StructField("deviceId", T.StringType(), False),
    T.StructField("timestamp", T.StringType(), True),
    T.StructField("epochMs", T.LongType(), True),
    T.StructField("updatePeriodSeconds", T.IntegerType(), True),
    T.StructField("readings", T.StructType([
        T.StructField("accelerometer", T.StructType([
            T.StructField("current", accel_current_schema, True),
            T.StructField("stats", accel_stats_schema, True)
        ]), True),
        T.StructField("gyroscope", T.StructType([
            T.StructField("current", gyro_current_schema, True),
            T.StructField("stats", gyro_stats_schema, True)
        ]), True)
    ]), True)
])

In [30]:
raw_stream_df = (spark.readStream
    .schema(sensor_payload_schema)
    .option("multiLine", True).json(input_file_1)
)
raw_stream_df.isStreaming

True

In [34]:
q1=raw_stream_df.writeStream.format("console").outputMode("update").option("truncate", "false").queryName("readings").start()
q1.awaitTermination(60)

In [38]:
q1.stop()

-------------------------------------------
Batch: 53
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



-------------------------------------------
Batch: 54
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



-------------------------------------------
Batch: 55
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



-------------------------------------------
Batch: 56
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



-------------------------------------------
Batch: 57
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



-------------------------------------------
Batch: 58
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



-------------------------------------------
Batch: 59
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



-------------------------------------------
Batch: 60
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



# Fall and impact detection

In [37]:
# Convert epoch milliseconds to event timestamp with watermark
flattened_stream = (raw_stream_df
    .withColumn("event_time", F.to_timestamp(F.from_unixtime(F.col("epochMs") / 1000)))
    .withWatermark("event_time", "20 seconds")
    .select(
        F.col("deviceId"),
        F.col("event_time"),
        F.col("readings.accelerometer.stats.magnitudeMin").alias("accel_min_magnitude"),
        F.col("readings.accelerometer.stats.magnitudeMax").alias("accel_max_magnitude"),
        F.col("readings.accelerometer.stats.magnitudeVariance").alias("accel_variance"),
        F.col("readings.gyroscope.stats.peakRotationRate").alias("gyro_peak_rotation")
    )
)


# 10-second tumbling window grouped by deviceId
windowed_aggregates = (flattened_stream
    .groupBy(
        F.col("deviceId"),
        F.window(F.col("event_time"), "10 seconds", "10 seconds")
    )
    .agg(
        F.min("accel_min_magnitude").alias("window_min_accel"),
        F.max("accel_max_magnitude").alias("window_max_accel"),
        F.avg("accel_variance").alias("window_avg_variance"),
        F.max("gyro_peak_rotation").alias("window_max_rotation"),
        F.count("*").alias("batch_count")
    )
)

query_impact = (windowed_aggregates
    .filter(F.col("window_max_accel") >= 24.0)
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("deviceId"),
        F.lit("HIGH_IMPACT_DETECTED").alias("anomaly_type"),
        F.round(F.col("window_max_accel"), 2).alias("peak_impact_ms2"),
        F.when(F.col("window_max_accel") >= 40.0, "SEVERE_COLLISION")
         .when(F.col("window_max_accel") >= 30.0, "HARD_IMPACT")
         .otherwise("MODERATE_IMPACT").alias("impact_class")
    )
    .writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", "false")
    .queryName("query_impact_detection")
    .start()
)
query_impact.awaitTermination(60)

-------------------------------------------
Batch: 38
-------------------------------------------
+----------------------+------------------------+-------------+-------------------+--------------------------------------------------------------------------------------------------------------------------------------------+
|deviceId              |timestamp               |epochMs      |updatePeriodSeconds|readings                                                                                                                                    |
+----------------------+------------------------+-------------+-------------------+--------------------------------------------------------------------------------------------------------------------------------------------+
|android-sensor-hanisaf|2026-10-18T19:51:44.982Z|1792353104982|10                 |{{{0.0, 0.2, 9.8, 9.802}, {500, 0.0158, 0.1998, 9.8, 9.805, 9.801, 0.0, false}}, {{0.0, 0.0, 0.0, 0.0}, {500, 0.003, 0.001, 0.0018, 0.316}}}|
+-

-------------------------------------------
Batch: 39
-------------------------------------------
+------------------------+------------------------+-------------+-------------------+------------+
|deviceId                |timestamp               |epochMs      |updatePeriodSeconds|readings    |
+------------------------+------------------------+-------------+-------------------+------------+
|android-sensor-de3d-43ab|2026-10-18T19:51:48.084Z|1792353108084|10                 |{NULL, NULL}|
+------------------------+------------------------+-------------+-------------------+------------+

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------------------+------------------------+--------------------+---------------+----------------+
|window_start       |window_end         |deviceId                |anomaly_type        |peak_impact_ms2|impact_class    |
+-------------------+-------------------+------------------------+-

-------------------------------------------
Batch: 1
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+

-------------------------------------------
Batch: 40
-------------------------------------------
+----------------------+------------------------+-------------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|deviceId              |timestamp               |epochMs      |updatePeriodSeconds|readings                                                                                                                                                        |
+-----

-------------------------------------------
Batch: 3
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



...

-------------------------------------------
Batch: 43
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



-------------------------------------------
Batch: 44
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+

-------------------------------------------
Batch: 64
-------------------------------------------
+----------------------+------------------------+-------------+-------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------+
|deviceId              |timestamp               |epochMs      |updatePeriodSeconds|readings                                                                                                                                                 |
+------------------

-------------------------------------------
Batch: 45
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



-------------------------------------------
Batch: 46
-------------------------------------------
+------------+----------+--------+------------+---------------+------------+
|window_start|window_end|deviceId|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+------------+---------------+------------+
+------------+----------+--------+------------+---------------+------------+



[Stage 194:=========>  (154 + 32) / 200][Stage 195:>                (0 + 0) / 1]

In [39]:
query_impact.stop()

26/10/18 19:54:55 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 61, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
26/10/18 19:54:55 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 61, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.
26/10/18 19:54:55 ERROR Utils: Aborting task
org.apache.spark.SparkException: [CANNOT_WRITE_STATE_STORE.CANNOT_COMMIT] Error writing state store files for provider HDFSStateStore[stateStoreId=StateStoreId[ operatorId=0, partitionId=35, storeName=default ], version=61]. Cannot perform commit during state checkpoint. SQLSTATE: 58030
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedToCommitStateFileError(QueryExecutionErrors.scala:2270)
	at org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider$HDFSBackedStateStore.commit(HDFSBackedStateStoreProvider.scala:197)
	at org.apache.spark.sql.execution.streaming.operators.stateful.StreamingAg

In [40]:
spark.stop()